# General Data Science Track — Day 2: Systematic cleaning and validation
**Estimated core time:** 60 minutes  
**Curriculum:** Phase 1.2 — Selection, indexing, assignment, and data quality

## Why this matters
A customer import can look plausible while containing duplicated people, inconsistent labels, and ages that should never enter a summary. Your question today: **Which records can we safely retain, and how much age information remains usable?** Build a small, auditable cleaning workflow and explain its limits. This session builds on your completed Day 1; it runs independently of that notebook.

## Learning objectives
By the end, you will be able to:
- predict Series versus DataFrame results and distinguish labels from positions;
- select and assign safely with Boolean masks and `.loc`;
- distinguish missing, unparseable, and out-of-range values;
- normalize strings and quarantine conflicting customer records; and
- enforce a data contract with assertions and report completeness at customer grain.

## R-to-Python bridge
| R / tidyverse | pandas |
|---|---|
| `select(age)` | `df[["age"]]` gives a DataFrame; `df["age"]` gives a Series |
| `filter()` | `df.loc[mask, columns]` |
| Row numbers | `.iloc` uses zero-based positions; `.loc` uses index labels |
| `mutate()` | `df["new"] = values` or `df.loc[mask, "column"] = value` |
| `&`, `|`, `!` | `&`, `|`, `~`; parenthesize each pandas comparison |
| `str_trim()`, `str_to_lower()` | `.str.strip()`, `.str.lower()` |
| `factor()` | `pd.Categorical(..., categories=...)` |

A Series carries an index: assignment of a Series aligns **labels**, not merely row order. Python's `and` asks for a single truth value and cannot combine whole Boolean Series.

## Suggested pacing
- Tasks 1–3: load, select, deduplicate — 17 minutes
- Tasks 4–6: strings, numeric validity, assignment — 20 minutes
- Tasks 7–9: categories, assertions, reporting — 13 minutes
- Retrieval — 5 minutes
- Reasoning and reflection — 5 minutes

Keep explanations to a few sentences; the extension is outside this hour.

### Inputs and cleaning policy
Open this notebook with the **repository root as the working directory**. Use `data/customer_import.csv`, a fixed synthetic teaching fixture created by `agent/generate_learning_data.py`. It contains 12 import rows and 5 columns, representing 10 intended customers. These `IMP...` IDs are independent of Day 1's customer IDs. No joins or downloads are needed.

- `customer_id` is the intended customer key; the raw import can repeat it.
- Preserve the original raw table and the three `*_raw` columns.
- Remove exact duplicate rows first. Then quarantine **all** remaining rows for any repeated customer key; no source timestamp establishes which version is correct.
- Normalize whitespace and case; standardize segment spaces to underscores.
- Allowed regions: `north`, `south`, `east`, `west`. Other labels become missing in the cleaned column and retain an invalidity flag.
- Allowed segments: `consumer`, `small_business`, `enterprise`; these are unordered.
- For this adult-customer exercise, usable ages are numeric values from 18 through 100 inclusive. Keep unusable ages missing; do not impute.
- An empty age field is missing at source; a nonempty token that cannot parse is a separate issue. CSV loading will interpret the empty field as missing.

These rules are an exercise data contract, not claims about universal age limits or region definitions.

## Task 1 — Load and predict object shapes (5 minutes)
Import pandas as `pd`. Read the input into `raw`, explicitly reading `age_raw` as pandas `"string"` via the `dtype` argument. Inspect shape, dtypes, and the first rows.

Before executing, predict the type and shape of each expression:
```python
raw["age_raw"]
raw[["age_raw"]]
raw.iloc[:3, :2]
```
Confirm your predictions. State the raw row grain and intended customer key. Check that `raw.shape` is `(12, 5)`.

API reminder: `pd.read_csv(path, dtype={column_name: "string"})`.

1. raw will return the column age raw as array
2. raw will be a dataframe with the column age_raw
3. will return first 3 rows and first t2o columns

In [20]:
import pandas as pd

raw = pd.read_csv("data/customer_import.csv")

In [21]:
raw["age_raw"]
raw[["age_raw"]]
raw.iloc[:3, :2]
raw.shape
# All is confirmed

(12, 5)

## Task 2 — Labels, positions, and Boolean selection (5 minutes)
Create `indexed` with `customer_id` as the index. Predict how many rows `indexed.loc[["IMP002"]]` returns, then compare it with `indexed.iloc[[1]]`. Explain why a label lookup need not return one row.

Using `raw`, select opted-in import rows whose age token is not missing. Return only `customer_id` and `age_raw`; do not yet interpret or clean the ages.

Methods: `.set_index()`, `.loc`, `.iloc`, `.notna()`. Hint: combine parenthesized Boolean conditions with `&`. Your selection must remain a two-column DataFrame; repeated customers are still allowed here.

In [22]:
# Make customer_id index of raw dataframe
indexed = raw.set_index("customer_id")

indexed.loc[["IMP002"]] should return exactly 1 row, if there is only one customer with this id. 

In [23]:
indexed.loc[["IMP002"]]

,age_raw,region_raw,segment_raw,marketing_opt_in
customer_id,,,,
IMP002,51,SOUTH,enterprise,False
IMP002,52,SOUTH,enterprise,False


It has two, and they differ by age. this is weird. maby ea duplicate? 

In [24]:
indexed.iloc[[1]]

,age_raw,region_raw,segment_raw,marketing_opt_in
customer_id,,,,
IMP002,51,SOUTH,enterprise,False


This returned the second row of the dataset, first one is 0.

In [ ]:
# Select
indexed_notna = indexed.dropna(subset=["age_raw"])
indexed_notna["age_raw"].unique()
# This will require cleaning, too old negative ages, and non-numeric values

<StringArray>
['34', '51', 'unknown', '-4', '145', ' 42 ', '29', '63', '38', '52']
Length: 10, dtype: str

In [37]:
# now from raw, select all customers with non-missing age and who opted in on marketing
selected = raw.loc[raw["age_raw"].notna() & raw["marketing_opt_in"], ["customer_id", "age_raw"]]
selected.head()

,customer_id,age_raw
0,IMP001,34
4,IMP005,-4
6,IMP007,42
8,IMP009,63
10,IMP001,34


## Task 3 — Separate duplicate rows from conflicting customers (7 minutes)
Count exact duplicate rows, then create `deduped` by dropping those exact repetitions. On `deduped`, flag every row belonging to a repeated `customer_id` with `.duplicated(subset=..., keep=False)`.

Create `quarantine` from flagged rows and `clean` from the others; use `.copy()` when creating both. Inspect the conflicting values and state why keeping the first customer record would be arbitrary.

Acceptance criteria: 1 exact duplicate removed, 2 rows quarantined, 9 customers retained. `clean` has a unique, nonmissing key. Its index labels may have gaps: do not reset them yet.

Hint: `.drop_duplicates()` compares all columns by default; deduplicating only the customer key would erase the conflict evidence.

In [44]:
raw.duplicated().sum()
# THERE IS one exact duplicated row.

np.int64(1)

In [45]:
deduped = raw.drop_duplicates()

In [56]:
deduped["is_duplicated_id"] = deduped.duplicated(subset=["customer_id"], keep=False)
quarantined = deduped.loc[deduped["is_duplicated_id"] == True]
quarantined.head()

clean = deduped.loc[deduped["is_duplicated_id"] == False]
clean.head(100)

,customer_id,age_raw,region_raw,segment_raw,marketing_opt_in,is_duplicated_id
0,IMP001,34,North,Consumer,True,False
2,IMP003,NaN,east,small business,True,False
3,IMP004,unknown,West,CONSUMER,False,False
4,IMP005,-4,north,consumer,True,False
5,IMP006,145,central,enterprise,False,False
6,IMP007,42,EAST,small_business,True,False
7,IMP008,29,south,consumer,False,False
8,IMP009,63,west,enterprise,True,False
9,IMP010,38,north,consumer,False,False


## Task 4 — Normalize strings while preserving evidence (7 minutes)
On `clean`, create `region` and `segment` from their raw counterparts. Strip surrounding whitespace and lowercase both; replace spaces with underscores in `segment` using `.str.replace(" ", "_", regex=False)`.

Add Boolean `region_invalid` for normalized regions outside the allowed set. Use `.loc` to set only those invalid cleaned regions to `pd.NA`. Retain the raw values. Inspect distinct cleaned labels and the flagged row.

Methods: `.str.strip()`, `.str.lower()`, `.isin()`. Hint: `~` reverses a Boolean mask. Do this before categorical conversion so unexpected labels remain visible during the audit.

Acceptance criteria: one invalid region, four recognized region labels, three normalized segment labels, and unchanged row count.

In [68]:
region_valid = clean.copy()
region_valid["region"] = region_valid["region_raw"].str.replace(" ", "_", regex=False)
# remove all "_" in the region column
region_valid["region"] = region_valid["region"].str.strip("_")
# make all letters lowercase
region_valid["region"] = region_valid["region"].str.lower()
# set all values that are not north, south, east, or west to NaN
region_valid.loc[~region_valid["region"].isin(["north", "south", "east", "west"]), "region"] = None
region_valid

,customer_id,age_raw,region_raw,segment_raw,marketing_opt_in,is_duplicated_id,region
0,IMP001,34,North,Consumer,True,False,north
2,IMP003,NaN,east,small business,True,False,east
3,IMP004,unknown,West,CONSUMER,False,False,west
4,IMP005,-4,north,consumer,True,False,north
5,IMP006,145,central,enterprise,False,False,NaN
6,IMP007,42,EAST,small_business,True,False,east
7,IMP008,29,south,consumer,False,False,south
8,IMP009,63,west,enterprise,True,False,west
9,IMP010,38,north,consumer,False,False,north


## Task 5 — Distinguish three reasons an age is unusable (8 minutes)
Create `age` with `pd.to_numeric(clean["age_raw"], errors="coerce")`. Add three nonmissing Boolean flags:
- `age_missing_source`: raw token was missing;
- `age_parse_failed`: raw token was present but numeric conversion produced missing;
- `age_out_of_range`: numeric value is present but outside the allowed interval.

Inspect customer ID, raw token, parsed age, and all flags for affected rows. Do **not** replace out-of-range ages until Task 6.

Methods: `.isna()`, `.notna()`, `.between(18, 100)`, `.fillna(False)`. Hint: nullable strings can produce nullable Boolean results; ensure each final flag is strictly True or False.

Acceptance criteria: the three flag counts are 1, 1, and 2. Explain why `errors="coerce"` without a parse-failure audit could silently hide a source problem.

In [96]:
age_clean = region_valid.copy()

age_clean["age"] = pd.to_numeric(age_clean["age_raw"], errors="coerce")
age_clean["age_missing_source"] = age_clean["age_raw"].isna()
age_clean["age_parse_failed"] = age_clean["age"].isna() & ~age_clean["age_raw"].isna()
age_clean["age_out_of_range"] = ((age_clean["age"] < 0) | (age_clean["age"] > 120))


age_clean
clean = age_clean.copy()
clean

,customer_id,age_raw,region_raw,segment_raw,marketing_opt_in,is_duplicated_id,region,age,age_missing_source,age_parse_failed,age_out_of_range
0,IMP001,34,North,Consumer,True,False,north,34.0,False,False,False
2,IMP003,NaN,east,small business,True,False,east,NaN,True,False,False
3,IMP004,unknown,West,CONSUMER,False,False,west,NaN,False,True,False
4,IMP005,-4,north,consumer,True,False,north,-4.0,False,False,True
5,IMP006,145,central,enterprise,False,False,NaN,145.0,False,False,True
6,IMP007,42,EAST,small_business,True,False,east,42.0,False,False,False
7,IMP008,29,south,consumer,False,False,south,29.0,False,False,False
8,IMP009,63,west,enterprise,True,False,west,63.0,False,False,False
9,IMP010,38,north,consumer,False,False,north,38.0,False,False,False


## Task 6 — Debug assignment and index alignment (5 minutes)
An AI suggests:
```python
clean[clean["age_out_of_range"]]["age"] = pd.NA
```
Explain which object this assignment targets and why it does not reliably update `clean` (pandas 3 uses Copy-on-Write). Write a single `.loc` assignment that sets the flagged ages missing in `clean`. Check that exactly four ages are now missing and all remaining ages meet the contract.

Now predict this unrelated example **without running it first**, then verify your prediction:
```python
probe = pd.DataFrame({"value": [10, 20]}, index=[3, 8])
probe["replacement"] = pd.Series([70, 80])
```
Why do the assigned values become missing? Describe how matching index labels would change the result. The `.loc` mask for your age fix should come from `clean` itself.

In [131]:
age_clean.loc[age_clean["age_out_of_range"] == True, "age_raw"] = pd.NA
age_clean


# if any of the age flags are True, set he corresponding age values to pd.NA
age_clean.loc[(age_clean["age_missing_source"] == True) | (age_clean["age_out_of_range"] == True) | (age_clean["age_parse_failed"] == True), "age"] = pd.NA
age_clean


clean = age_clean.copy()

In [93]:
probe = pd.DataFrame({"value": [10, 20]}, index=[3, 8])
probe["replacement"] = pd.Series([70, 80])
probe
# The replacments are missing because the indices of the replacement series do not match the indices of the original DataFrame.

,value,replacement
3,10,NaN
8,20,NaN


## Task 7 — Convert categories deliberately (3 minutes)
Convert cleaned `region` and `segment` to categorical columns with their explicit allowed levels and `ordered=False`. Check that conversion introduces no additional missing values. Display category levels and counts, including missing values.

API reminder: `pd.Categorical(series, categories=allowed_labels, ordered=False)`; `.value_counts(dropna=False)`.

Explain why alphabetical display order does not imply a meaningful ranking of customer segments.

In [132]:
# First clean segment categories
clean["segment"] = clean["segment_raw"].str.replace(" ", "_").str.strip("_").str.lower()

In [133]:

allowed_segments = ["consumer", "small_business", "enterprise"]
allowed_regions = ["north", "south", "east", "west"]

pd.Categorical(clean["segment"], categories=allowed_segments, ordered=False)
pd.Categorical(clean["region"], categories=allowed_regions, ordered=False)

clean.reset_index(drop=True, inplace=True)
clean

,customer_id,age_raw,region_raw,segment_raw,marketing_opt_in,is_duplicated_id,region,age,age_missing_source,age_parse_failed,age_out_of_range,segment
0,IMP001,34,North,Consumer,True,False,north,34.0,False,False,False,consumer
1,IMP003,NaN,east,small business,True,False,east,NaN,True,False,False,small_business
2,IMP004,NaN,West,CONSUMER,False,False,west,NaN,False,True,False,consumer
3,IMP005,NaN,north,consumer,True,False,north,NaN,False,False,True,consumer
4,IMP006,NaN,central,enterprise,False,False,NaN,NaN,False,False,True,enterprise
5,IMP007,42,EAST,small_business,True,False,east,42.0,False,False,False,small_business
6,IMP008,29,south,consumer,False,False,south,29.0,False,False,False,consumer
7,IMP009,63,west,enterprise,True,False,west,63.0,False,False,False,enterprise
8,IMP010,38,north,consumer,False,False,north,38.0,False,False,False,consumer


## Task 8 — Turn the cleaning policy into assertions (5 minutes)
Write assertions that check:
1. 9 retained rows, a nonmissing unique customer key;
2. every observed age falls in the allowed range, and exactly four ages are missing;
3. the union of the three age flags equals the cleaned age-missing mask, row by row;
4. region missingness equals `region_invalid`, row by row, and segment has no missing values;
5. raw import rows = exact duplicate rows removed + quarantined rows + retained rows;
6. raw columns on retained index labels still equal the corresponding rows in `raw`.

Reminder using unrelated data:
```python
assert scores.dropna().between(0, 10).all(), "Observed scores outside the contract"
```
Hint: `.equals()` compares labeled pandas objects; `.all()` reduces a Boolean Series to one truth value. Explicitly permit missing ages rather than treating every missing value as a failed range check.

In [135]:
clean

,customer_id,age_raw,region_raw,segment_raw,marketing_opt_in,is_duplicated_id,region,age,age_missing_source,age_parse_failed,age_out_of_range,segment
0,IMP001,34,North,Consumer,True,False,north,34.0,False,False,False,consumer
1,IMP003,NaN,east,small business,True,False,east,NaN,True,False,False,small_business
2,IMP004,NaN,West,CONSUMER,False,False,west,NaN,False,True,False,consumer
3,IMP005,NaN,north,consumer,True,False,north,NaN,False,False,True,consumer
4,IMP006,NaN,central,enterprise,False,False,NaN,NaN,False,False,True,enterprise
5,IMP007,42,EAST,small_business,True,False,east,42.0,False,False,False,small_business
6,IMP008,29,south,consumer,False,False,south,29.0,False,False,False,consumer
7,IMP009,63,west,enterprise,True,False,west,63.0,False,False,False,enterprise
8,IMP010,38,north,consumer,False,False,north,38.0,False,False,False,consumer


In [143]:
# assert that clean data has 9 rows.
assert clean.shape[0] == 9, "Clean data should have 9 rows"

# age is within allowed range and exactl has four nas
assert clean["age"].dropna().between(1, 120).all(), "All non-missing ages should be within the allowed range"

assert clean["age"].isna().sum() == 4, "There should be exactly four missing age values"

## Task 9 — Report usable information at customer grain (5 minutes)
From `clean`, produce one row per segment with retained customer count, nonmissing age count, and median usable age. Use named aggregation and explicitly set `observed=True` in the groupby. Add age completeness as a percentage using retained customers as the denominator.

State one numerical completeness finding and one limitation in plain language. Distinguish `.size()` from `.count()` when an age is missing.

Acceptance criteria: 3 segment rows; customer counts sum to 9 and usable age counts sum to 5. No duplicated or quarantined customers contribute. This is a tiny synthetic import: a segment difference does not establish a population pattern.

In [ ]:
segment_summary = clean.groupby("segment",observed=True).agg(
    customer_count = ("customer_id", "count"),
    non_missing_age_count = ("age", lambda x: x.notna().sum()),
    median_age = ("age", "median")
).reset_index()

segment_summary["age_completeness"] = segment_summary["non_missing_age_count"] / segment_summary["customer_count"] * 100
segment_summary

,segment,customer_count,non_missing_age_count,median_age,age_completeness
0,consumer,5,3,34.0,60.0
1,enterprise,2,1,63.0,50.0
2,small_business,2,1,42.0,50.0


## Retrieval practice — Task 10: select, filter, and sort (5 minutes)
From memory, return the three oldest retained customers with usable ages, in descending age order. Display exactly `customer_id`, `region`, `segment`, and `age` in that order. Add assertions for row count and descending ages.

State the result's type and shape. Use the cleaned ages; leave `clean` unchanged.

In [158]:
clean.shape

(9, 12)

In [172]:
oldest = clean.sort_values("age", ascending=False).head(3)[["customer_id","region","segment", "age"]]



assert oldest.shape[0] == 3, "There should be exactly 3 oldest customers"
assert oldest.iloc[0,3] >= oldest.iloc[1,3] >= oldest.iloc[2,3], "The ages should be in descending order"

np.float64(63.0)

## Professional reasoning questions (3 minutes)
Answer briefly in a new markdown cell, or use the empty cell below to print a multiline response.
1. Explain why quarantining both conflicting records is more defensible under this contract than automatically selecting the larger age. What source evidence could resolve the conflict?
2. What could go wrong if you converted raw region strings to categories before inspecting unknown labels?
3. Why might the median of usable ages misrepresent a segment, even after every assertion passes? Name one conclusion this import cannot justify.

## Self-check and reflection (2 minutes)
Record in a new markdown cell, or print a multiline response below:
- Confidence from 1–5
- Hardest point
- One thing you can now explain
- One operation that deserves repetition
- Whether the hour felt too light, about right, or too heavy

Finish line: your cleaning cells run from a fresh kernel, assertions pass, and you can explain both retained and excluded information. Ask for review when ready.

## Optional extension — Transfer the audit to Day 1's customers
**Outside the estimated core duration (15–20 minutes).** Load `data/customers.csv` independently. At one-row-per-customer grain, audit key uniqueness, allowed region/segment labels, and the same adult-age rule. Build a compact table of failed checks and missing-age counts by segment. Preserve raw ages and do not impute.

Acceptance criteria: 2,000 customer rows; report issues actually observed rather than assuming this file has the import fixture's errors. Explain which checks transfer unchanged and which depend on the source's data contract.